###1. Kết nối bảng theo 2 cách

In [1]:
import sqlite3
import pandas as pd
# Tạo database trong RAM
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()
# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")
# Insert dữ liệu student
students = [
(1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
(2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
(3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
(4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
(5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
(6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
(7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
(8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
(9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
(10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)
# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER,
    course_name TEXT
)
""")
# Insert dữ liệu course
courses = [
(12, 'Giai tich'),
(34, 'Thong ke'),
(26, 'Tin hoc')
]
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)
conn.commit()

Kết nối bảng bằng tích Decartes

In [2]:
#Kết nối bảng bằng tích decartes
df_cross = pd.read_sql_query("""
SELECT *
FROM student
CROSS JOIN course
""", conn)
df_cross

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
8,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Nhận xét:
- Phép tích Descartes (CROSS JOIN) tạo ra tất cả các tổ hợp giữa hai bảng. Kết quả thu được có số dòng bằng tích số dòng của hai bảng ban đầu. Trong trường hợp này, mỗi sinh viên được kết hợp với tất cả các môn học, không phụ thuộc vào course_id.
- Phép nối này thường ít được sử dụng trong thực tế do tạo ra nhiều dữ liệu dư thừa, nhưng có ý nghĩa trong việc minh họa nguyên lý kết hợp dữ liệu giữa các bảng.

Kết nối bảng bằng JOIN

In [3]:
#Kết nối bảng bằng INNER JOIN
df_inner = pd.read_sql_query("""
SELECT *
FROM student
INNER JOIN course
ON student.course_id = course.id;
""", conn)
df_inner

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,34,Thong ke


Nhận xét:
- INNER JOIN chỉ trả về các bản ghi có khóa liên kết tồn tại ở cả hai bảng. Trong trường hợp này, chỉ những sinh viên có course_id trùng với id trong bảng course mới được giữ lại, do đó số lượng bản ghi giảm xuống còn 3.
- Các bản ghi có course_id không tồn tại hoặc bị NULL sẽ bị loại bỏ trong phép INNER JOIN.

In [4]:
#Kết nối bảng bằng LEFT JOIN
df_left = pd.read_sql_query("""
SELECT *
FROM student
LEFT JOIN course
ON student.course_id = course.id;
""", conn)
df_left

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,NaN,None


Nhận xét:
- LEFT JOIN trả về toàn bộ các bản ghi của bảng student và các bản ghi tương ứng ở bảng course nếu có. Trong trường hợp không tìm thấy bản ghi phù hợp, các giá trị từ bảng course sẽ được thay bằng NULL.
- Kết quả cho thấy một số sinh viên có course_id không tồn tại hoặc bị NULL, do đó không thể liên kết với bảng course và xuất hiện giá trị NULL ở các cột của bảng course.



In [5]:
#Kết nối bảng bằng RIGHT JOIN (được mô phỏng bằng LEFT JOIN)
df_right = pd.read_sql_query("""
SELECT s.*, c.id, c.course_name
FROM course c
LEFT JOIN student s
ON s.course_id = c.id;
""", conn)
df_right

,student_id,name,class,course_id,score,id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke
3,NaN,None,None,NaN,NaN,26,Tin hoc


Nhận xét:
- Kết quả của phép nối cho thấy dữ liệu được giữ lại đầy đủ từ bảng course, bao gồm cả những môn học không có sinh viên đăng ký.

Cụ thể:

- Các môn học có sinh viên đăng ký như Giải tích (id = 12) và Thống kê (id = 34) được hiển thị kèm thông tin sinh viên tương ứng.
Môn học Tin học (id = 26) không có sinh viên nào đăng ký nên các trường thông tin của bảng student (student_id, name, class, score,…) đều có giá trị NULL.

In [6]:
#Kết nối bảng bằng FULL OUTER JOIN (được mô phỏng bằng UNION ALL)
df_full = pd.read_sql_query("""
SELECT *
FROM student s
LEFT JOIN course c ON s.course_id = c.id
UNION
SELECT *
FROM course c
LEFT JOIN student s ON s.course_id = c.id
""", conn)
df_full

,student_id,name,class,course_id,score,id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,12.0,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,34.0,Thong ke
2,3,Pham Van Nam,Toan Tin,None,7.9,NaN,None
3,4,Le Thanh Huyen,Toan Tin,20,7.2,NaN,None
4,5,Vu Quoc Anh,May Tinh,24,8.0,NaN,None
5,6,Dang Thuy Linh,May Tinh,24,5.5,NaN,None
6,7,Bui Tien Dung,Kinh Te,34,9.2,34.0,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20,8.8,NaN,None
8,9,Duong Huu Phuc,Kinh Te,None,7.2,NaN,None
9,10,Cao Thi Hanh,May Tinh,None,7.0,NaN,None


Nhận xét:
- FULL OUTER JOIN trả về toàn bộ dữ liệu của cả hai bảng student và course, bao gồm cả các bản ghi không có liên kết tương ứng. Kết quả cho thấy có những sinh viên không có môn học và những môn học không có sinh viên đăng ký.
- Do SQLite không hỗ trợ trực tiếp FULL OUTER JOIN, nên kết quả được tạo ra bằng cách kết hợp LEFT JOIN và RIGHT JOIN thông qua UNION. Điều này có thể dẫn đến việc xuất hiện các bản ghi trùng lặp.

###2. Cập nhật những giá trị Course_ID còn thiếu

In [7]:
cursor.execute("""
UPDATE student
SET course_id = CASE
    WHEN class = 'Toan Tin' THEN 26
    WHEN class = 'Kinh Te' THEN 34
    WHEN class = 'May Tinh' THEN 12
END
WHERE course_id IS NULL
""")
conn.commit()
#Loại bỏ bản ghi tham gia những môn học không tồn tại
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()
pd.read_sql_query("SELECT * FROM student", conn)

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,34,7.2
5,10,Cao Thi Hanh,May Tinh,12,7.0


Nhận xét:
- Ban đầu bảng student có 10 bản ghi, sau khi thực hiện cập nhật và làm sạch dữ liệu thì còn lại 6 bản ghi hợp lệ.
- Sau khi xử lý, số lượng bản ghi giảm từ 10 xuống còn 6, trong đó 4 bản ghi không hợp lệ đã bị loại bỏ. Đồng thời, 3 bản ghi có course_id bị thiếu đã được cập nhật thành các giá trị hợp lệ. Kết quả cuối cùng đảm bảo 100% dữ liệu có thể liên kết với bảng course, giúp nâng cao tính chính xác và toàn vẹn dữ liệu.

In [8]:
#Tổng số sinh viên, điểm trung bình của từng lớp
pd.read_sql_query("""
SELECT class,
       COUNT(*) AS so_sv,
       AVG(score) AS diem_tb
FROM student
GROUP BY class
""", conn)

,class,so_sv,diem_tb
0,Kinh Te,3,8.533333
1,May Tinh,2,6.850000
2,Toan Tin,1,7.900000


Nhận xét:
- Từ bảng số liệu có thể thấy tổng số sinh viên của ba lớp là 6 sinh viên, trong đó lớp Kinh Tế có số lượng sinh viên nhiều nhất (3 sinh viên), tiếp theo là lớp Máy Tính (2 sinh viên) và ít nhất là lớp Toán Tin (1 sinh viên). Điều này cho thấy sự phân bố số lượng sinh viên giữa các lớp chưa đồng đều.
- Về kết quả học tập, lớp Kinh Tế có điểm trung bình cao nhất với 8.53, cho thấy chất lượng học tập của lớp này khá tốt và ổn định. Lớp Toán Tin có điểm trung bình là 7.90, ở mức khá, tuy nhiên do số lượng sinh viên ít nên kết quả này chưa phản ánh đầy đủ mặt bằng chung. Trong khi đó, lớp Máy Tính có điểm trung bình thấp nhất (6.85), cho thấy cần có sự cải thiện trong quá trình học tập của sinh viên lớp này.
- Nhìn chung, có thể nhận thấy sự chênh lệch về cả số lượng sinh viên và kết quả học tập giữa các lớp. Điều này cho thấy cần có các biện pháp hỗ trợ phù hợp nhằm nâng cao chất lượng học tập, đặc biệt đối với những lớp có điểm trung bình thấp hơn.

In [9]:
#Tổng số sinh viên, điểm trung bình của từng môn học
pd.read_sql_query("""
SELECT c.course_name,
       COUNT(*) AS so_sv,
       AVG(s.score) AS diem_tb
FROM student s
JOIN course c
ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,so_sv,diem_tb
0,Giai tich,2,6.850000
1,Thong ke,3,8.533333
2,Tin hoc,1,7.900000


Nhận xét:
- Kết quả cho thấy có 3 môn học, với tổng số 6 sinh viên được phân bổ và điểm trung bình khác nhau giữa các môn.
- Kết quả thống kê cho thấy môn Thống kê có số lượng sinh viên nhiều nhất (3/6) và điểm trung bình cao nhất (8.53), trong khi môn Giải tích có điểm trung bình thấp nhất (6.85). Môn Tin học có số lượng sinh viên ít nhất (1 sinh viên) với mức điểm trung bình khá. Điều này cho thấy sự khác biệt rõ rệt về cả số lượng và kết quả học tập giữa các môn.


In [10]:
#Phân loại thi đua theo điểm số của từng môn học
pd.read_sql_query("""
SELECT c.course_name,
       AVG(s.score) AS diem_tb,
       CASE
           WHEN AVG(s.score) >= 9 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6 AND AVG(s.score) < 9 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai
FROM student s
JOIN course c
ON s.course_id = c.id
GROUP BY c.course_name
""", conn)

,course_name,diem_tb,xep_loai
0,Giai tich,6.850000,Tot
1,Thong ke,8.533333,Tot
2,Tin hoc,7.900000,Tot


Nhận xét:
- Kết quả cho thấy cả 3 môn học đều có điểm trung bình nằm trong khoảng từ 6 đến 8.9 nên được xếp loại “Tốt”, chiếm 100% tổng số môn. Điểm trung bình dao động từ 6.85 đến 8.53 với độ chênh lệch 1.68 điểm, cho thấy chất lượng học tập giữa các môn tương đối đồng đều và không có môn nào ở mức yếu hoặc xuất sắc.
- Mặc dù các môn đều được xếp loại “Tốt”, nhưng vẫn có sự chênh lệch về điểm trung bình, trong đó môn Thống kê có kết quả cao nhất (8.53) và Giải tích thấp nhất (6.85), cho thấy sự khác biệt về mức độ học tập giữa các môn.

###3. Xếp hạng sinh viên thông qua:
- Điểm số

- Điểm số theo lớp học

- Điểm số theo mã môn học

In [11]:
#Theo điểm số
pd.read_sql_query("""
SELECT *,
       RANK() OVER (ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3
3,9,Duong Huu Phuc,Kinh Te,34,7.2,4
4,10,Cao Thi Hanh,May Tinh,12,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6


Nhận xét:
- Kết quả xếp hạng cho thấy có 6 sinh viên được phân loại theo điểm số, trong đó 2 sinh viên đạt điểm cao nhất (9.2) và cùng xếp hạng 1, chiếm 33.3%. Do sử dụng hàm RANK(), thứ hạng 2 không xuất hiện và chuyển sang hạng 3 cho sinh viên tiếp theo. Điểm số dao động từ 6.7 đến 9.2 với độ chênh lệch 2.5 điểm, cho thấy sự phân hóa tương đối rõ giữa các sinh viên.

In [12]:
#Điểm số theo lớp học
pd.read_sql_query("""
SELECT *,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Nhận xét:
- Kết quả xếp hạng theo lớp cho thấy sinh viên được phân loại độc lập trong từng lớp học. Lớp Kinh Tế có số lượng sinh viên nhiều nhất (3 sinh viên) và xuất hiện hiện tượng đồng hạng cao nhất. Lớp Máy Tính có sự chênh lệch điểm nhỏ giữa các sinh viên, trong khi lớp Toán Tin chỉ có một sinh viên nên không có sự so sánh. Điều này phản ánh rõ sự khác biệt về quy mô và kết quả học tập giữa các lớp.

In [13]:
#Điểm số theo mã môn học
pd.read_sql_query("""
SELECT *,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
FROM student
""", conn)

,student_id,name,class,course_id,score,rank
0,10,Cao Thi Hanh,May Tinh,12,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,3,Pham Van Nam,Toan Tin,26,7.9,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3


Nhận xét:
- Kết quả xếp hạng theo mã môn học cho thấy sự khác biệt rõ rệt về kết quả học tập giữa các môn. Môn Thống kê có số lượng sinh viên nhiều nhất và xuất hiện đồng hạng cao nhất với 2 sinh viên đạt điểm 9.2. Môn Giải tích có mức điểm trung bình và chênh lệch nhỏ, trong khi môn Tin học chỉ có một sinh viên nên không có sự so sánh. Điều này cho thấy mức độ cạnh tranh và phân bố điểm khác nhau giữa các môn học.

In [14]:
#Xếp hạng Top 3 cao nhất theo điểm số
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score DESC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,3,Pham Van Nam,Toan Tin,26,7.9,3


Nhận xét:
- Kết quả xếp hạng Top 3 cho thấy hai sinh viên đạt điểm cao nhất (9.2) và cùng xếp hạng 1, chiếm 66.7% trong nhóm. Sinh viên xếp hạng tiếp theo có điểm 7.9, thấp hơn 1.3 điểm so với nhóm dẫn đầu. Việc sử dụng hàm RANK() dẫn đến hiện tượng đồng hạng và bỏ qua hạng 2, phản ánh chính xác sự chênh lệch điểm số giữa các sinh viên.

In [15]:
#Xếp hạng Top 3 thấp nhất theo điểm số
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (ORDER BY score ASC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3


Nhận xét:
- Kết quả cho thấy 3 sinh viên có điểm thấp nhất lần lượt là 6.7, 7.0 và 7.2. Sinh viên Nguyễn Minh Hoàng có điểm thấp nhất (6.7), tiếp theo là Cao Thị Hạnh (7.0) và Dương Hữu Phúc (7.2). Khoảng chênh lệch điểm giữa các sinh viên trong nhóm này là 0.5 điểm, cho thấy mức độ phân hóa không lớn ở nhóm điểm thấp.

In [16]:
#Xếp hạng Top 3 cao nhất của điểm số theo lớp học
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,1
2,9,Duong Huu Phuc,Kinh Te,34,7.2,3
3,10,Cao Thi Hanh,May Tinh,12,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Nhận xét:
- Kết quả cho thấy sự khác biệt rõ rệt về thành tích giữa các lớp. Lớp Kinh Tế có kết quả nổi bật nhất với 2 sinh viên đạt điểm cao nhất (9.2), trong khi lớp Máy Tính có mức điểm thấp hơn và chênh lệch nhỏ. Lớp Toán Tin chỉ có một sinh viên nên không có sự cạnh tranh. Điều này phản ánh quy mô và chất lượng học tập khác nhau giữa các lớp.

In [17]:
#Xếp hạng Top 3 thấp nhất của điểm số theo lớp học
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,9,Duong Huu Phuc,Kinh Te,34,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,2
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,12,7.0,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


Nhận xét:
- Lớp Máy Tính có điểm thấp nhất là 6.7 (Nguyễn Minh Hoàng), tiếp theo là 7.0 (Cao Thị Hạnh). Mức điểm dao động từ 6.7 → 7.0, cho thấy mặt bằng điểm của lớp này ở mức trung bình.
- Lớp Kinh Tế có điểm thấp nhất là 7.2 (Dương Hữu Phúc), cao hơn lớp Máy Tính. Hai sinh viên còn lại đều đạt 9.2, nên độ chênh lệch lớn (7.2 → 9.2), thể hiện sự phân hóa rõ trong lớp.
- Lớp Toán Tin chỉ có 1 sinh viên nên điểm thấp nhất cũng chính là 7.9 (Phạm Văn Nam). Không có sự so sánh nội bộ, nhưng mức điểm này cao hơn điểm thấp nhất của 2 lớp còn lại.

In [18]:
#Xếp hạng Top 3 cao nhất của điểm số theo mã môn học
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,10,Cao Thi Hanh,May Tinh,12,7.0,1
1,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
2,3,Pham Van Nam,Toan Tin,26,7.9,1
3,2,Tran Thi Lan,Kinh Te,34,9.2,1
4,7,Bui Tien Dung,Kinh Te,34,9.2,1
5,9,Duong Huu Phuc,Kinh Te,34,7.2,3


Nhận xét:
- Kết quả phân tích cho thấy số lượng sinh viên theo từng môn học không đồng đều: môn Thống kê có nhiều sinh viên nhất (3 sinh viên), trong khi Giải tích có 2 sinh viên và Tin học chỉ có 1 sinh viên. Về điểm trung bình, môn Thống kê đạt cao nhất (8.53), tiếp theo là Tin học (7.9) và thấp nhất là Giải tích (6.85).
- Khi phân tích chi tiết, sinh viên có điểm cao nhất là 9.2 (2 sinh viên), trong khi điểm thấp nhất là 6.7, cho thấy mức độ chênh lệch điểm là khá rõ (6.7 → 9.2). Xét theo từng lớp và từng môn học, lớp Kinh tế và môn Thống kê có kết quả nổi bật hơn, trong khi lớp Máy tính và môn Giải tích có mức điểm thấp hơn.
- Nhìn chung, dữ liệu cho thấy sự phân hóa rõ rệt về kết quả học tập giữa các môn và các lớp, đồng thời phản ánh sự khác biệt về chất lượng học tập của sinh viên trong từng nhóm.

In [19]:
#Xếp hạng Top 3 thấp nhất của điểm số theo mã môn học
pd.read_sql_query("""
SELECT *
FROM (
    SELECT *,
           RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank
    FROM student
)
WHERE rank <= 3
""", conn)

,student_id,name,class,course_id,score,rank
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,12,7.0,2
2,3,Pham Van Nam,Toan Tin,26,7.9,1
3,9,Duong Huu Phuc,Kinh Te,34,7.2,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,2
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


Nhận xét:
- Mã môn 12 (Máy Tính):
Hai sinh viên có điểm thấp nhất là 6.7 và 7.0.

→ Đây là môn có mức điểm thấp nhất trong tất cả các môn, cho thấy kết quả học tập còn hạn chế.
- Mã môn 26 (Toán Tin):
Chỉ có 1 sinh viên với điểm 7.9.

→ Mặc dù nằm trong danh sách thấp nhất nhưng điểm vẫn ở mức khá, không quá thấp.
Mã môn 34 (Kinh Tế):
- Ba sinh viên có điểm lần lượt 7.2, 9.2, 9.2.

→ Điểm thấp nhất là 7.2, cao hơn môn 12, cho thấy mặt bằng điểm của môn này tốt hơn. Tuy nhiên vẫn có sự chênh lệch lớn (7.2 → 9.2).

In [20]:
#Bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng Student
cursor.execute("""
ALTER TABLE student
ADD COLUMN graduation_date TEXT
""")

In [23]:
#Tính Graduation_date
cursor.execute("""
UPDATE student
SET graduation_date = DATETIME('now', '+' ||
    (SELECT rnk FROM (
        SELECT student_id,
               RANK() OVER (ORDER BY score DESC) AS rnk
        FROM student
    ) t
    WHERE t.student_id = student.student_id)
|| ' months')
""")
conn.commit()
pd.read_sql_query("SELECT student_id, score, graduation_date FROM student", conn)

,student_id,score,graduation_date
0,1,6.7,2026-10-03 00:33:23
1,2,9.2,2026-05-03 00:33:23
2,3,7.9,2026-07-03 00:33:23
3,7,9.2,2026-05-03 00:33:23
4,9,7.2,2026-08-03 00:33:23
5,10,7.0,2026-09-03 00:33:23


Nhận xét:
- Ngày tốt nghiệp được tính dựa trên điểm số, dao động từ tháng 05/2026 đến tháng 10/2026.
- Sinh viên có điểm cao (9.2) như student_id 2 và 7 có ngày tốt nghiệp sớm nhất (03/05/2026).

→ Cho thấy điểm càng cao thì thời gian hoàn thành càng sớm.
- Các sinh viên có điểm trung bình (7.0 – 7.9) có ngày tốt nghiệp vào khoảng 07/2026 – 09/2026.
- Sinh viên có điểm thấp nhất (6.7 – student_id 1) có ngày tốt nghiệp muộn nhất (03/10/2026).
Kết luận:
- Có mối quan hệ rõ ràng giữa điểm số và thời gian tốt nghiệp:

→ Điểm càng cao → tốt nghiệp càng sớm

→ Điểm thấp → tốt nghiệp muộn hơn
- Khoảng chênh lệch thời gian tốt nghiệp là khoảng 5 tháng (05/2026 → 10/2026), phản ánh sự khác biệt về kết quả học tập giữa các sinh viên.